# SipIt Inversion Test with Gemma 3 4B

Testing the SipIt algorithm from "Language Models are Injective and Hence Invertible" paper.
This notebook encodes a string, extracts hidden states, and tries to recover the original text.

In [ ]:
import sys
sys.path.insert(0, '../third_party/reproducibility-llm-inversion')

import torch
import numpy as np
from tqdm import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Using device: {device}")

# Check if unsloth is available
try:
    from unsloth import FastLanguageModel
    print("Unsloth available!")
    USE_UNSLOTH = True
except ImportError:
    print("Unsloth not installed. Install with: pip install unsloth")
    USE_UNSLOTH = False

In [ ]:
import torch.nn.functional as F

class SipItGemma3:
    """
    SipIt algorithm adapted for Gemma 3 models.
    Supports Unsloth pre-quantized bitsandbytes 4-bit models.
    """
    
    def __init__(self, model_name: str, layer_idx: int = -1, epsilon: float = 1e-5, max_seq_length: int = 2048):
        print(f"Loading model {model_name} with Unsloth...")
        
        # Load with Unsloth's FastLanguageModel (handles bnb-4bit properly)
        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_name,
            max_seq_length=max_seq_length,
            load_in_4bit=True,
            dtype=None,  # Auto-detect
        )
        
        # Put in inference mode
        FastLanguageModel.for_inference(self.model)
        
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        
        self.layer_idx = layer_idx
        self.epsilon = epsilon
        self.vocab_size = len(self.tokenizer)
        
        # Cache embedding matrix for gradient search
        self.embedding_matrix = self.model.get_input_embeddings().weight.data
        
        # Get device from model
        self.device = next(self.model.parameters()).device
        
        print(f"Model loaded! (Unsloth 4-bit)")
        print(f"Vocab size: {self.vocab_size}, Embedding dim: {self.embedding_matrix.shape[1]}")
        print(f"Device: {self.device}")
    
    def get_hidden_states(self, input_ids: torch.Tensor, layer_idx=None) -> torch.Tensor:
        """Extract hidden states from the specified layer."""
        layer = layer_idx if layer_idx is not None else self.layer_idx
        with torch.no_grad():
            input_ids = input_ids.to(self.device)
            outputs = self.model(input_ids, output_hidden_states=True)
            hidden_states = outputs.hidden_states[layer]
        return hidden_states
    
    def get_hidden_states_from_embeds(self, inputs_embeds: torch.Tensor, layer_idx=None) -> torch.Tensor:
        """Extract hidden states from embeddings (for gradient computation)."""
        layer = layer_idx if layer_idx is not None else self.layer_idx
        outputs = self.model(inputs_embeds=inputs_embeds, output_hidden_states=True)
        hidden_states = outputs.hidden_states[layer]
        return hidden_states
    
    def compute_hidden_state_for_candidate(self, prefix_ids: torch.Tensor, candidate_token: int, layer_idx=None) -> torch.Tensor:
        """Compute hidden state when appending candidate_token to prefix."""
        candidate_seq = torch.cat([
            prefix_ids,
            torch.tensor([[candidate_token]], device=self.device)
        ], dim=1)
        hidden_states = self.get_hidden_states(candidate_seq, layer_idx)
        return hidden_states[:, -1, :]
    
    def check_match(self, target_hidden: torch.Tensor, candidate_hidden: torch.Tensor, epsilon=None) -> bool:
        """Check if candidate hidden state matches target within epsilon-ball."""
        eps = epsilon if epsilon is not None else self.epsilon
        distance = torch.norm(target_hidden.float() - candidate_hidden.float(), p=2).item()
        return distance <= eps
    
    def gradient_guided_topk(self, prefix_ids: torch.Tensor, target_hidden: torch.Tensor, 
                              top_k: int = 256, layer_idx=None) -> list:
        """
        Use gradient to find top-k most promising token candidates.
        
        Returns: List of top-k token IDs sorted by gradient alignment
        """
        embedding_layer = self.model.get_input_embeddings()
        
        # Get prefix embeddings
        if prefix_ids.shape[1] > 0:
            with torch.no_grad():
                prefix_embeds = embedding_layer(prefix_ids)
        else:
            prefix_embeds = None
        
        # Create a learnable embedding for the candidate position
        embed_dim = self.embedding_matrix.shape[1]
        candidate_embed = torch.zeros(1, 1, embed_dim, device=self.device, dtype=torch.float32, requires_grad=True)
        
        # Concatenate prefix + candidate embedding
        if prefix_embeds is not None:
            inputs_embeds = torch.cat([prefix_embeds.float(), candidate_embed], dim=1)
        else:
            inputs_embeds = candidate_embed
        
        # Forward pass with gradients
        hidden_states = self.get_hidden_states_from_embeds(inputs_embeds, layer_idx)
        computed_hidden = hidden_states[:, -1:, :]
        
        # Loss = L2 distance to target
        loss = torch.norm(computed_hidden.float() - target_hidden.float(), p=2)
        
        # Backprop to get gradient
        loss.backward()
        
        grad = candidate_embed.grad.squeeze()
        
        # Find tokens aligned with negative gradient
        with torch.no_grad():
            scores = torch.matmul(self.embedding_matrix.float(), -grad.float())
            top_k_scores, top_k_indices = torch.topk(scores, k=min(top_k, self.vocab_size))
            
        return top_k_indices.tolist()
    
    def invert_sequence(self, target_hidden_states: torch.Tensor, policy: str = "random", 
                        max_iterations_per_position=None, top_k: int = 256, verbose: bool = False):
        """Invert a sequence from its hidden states using SipIt algorithm."""
        seq_len = target_hidden_states.shape[1]
        recovered_tokens = []
        distances = []
        
        if max_iterations_per_position is None:
            max_iterations_per_position = self.vocab_size
        
        prefix_ids = torch.empty((1, 0), dtype=torch.long, device=self.device)
        
        iterator = tqdm(range(seq_len), desc="Inverting tokens") if verbose else range(seq_len)
        
        for t in iterator:
            target_h_t = target_hidden_states[:, t:t+1, :]
            
            if policy == "random":
                vocab_order = list(range(self.vocab_size))
                np.random.shuffle(vocab_order)
            elif policy == "gradient":
                vocab_order = self.gradient_guided_topk(prefix_ids, target_h_t, top_k=top_k)
                if verbose:
                    print(f"\n  Gradient search: testing top {len(vocab_order)} candidates")
            else:
                raise ValueError(f"Unknown policy: {policy}")
            
            found = False
            best_distance = float('inf')
            best_token = None
            
            candidates_iter = vocab_order[:max_iterations_per_position]
            if verbose and policy == "random":
                candidates_iter = tqdm(candidates_iter, desc=f"Pos {t}", leave=False)
            
            for candidate_token in candidates_iter:
                candidate_h = self.compute_hidden_state_for_candidate(prefix_ids, candidate_token)
                distance = torch.norm(target_h_t.float() - candidate_h.float(), p=2).item()
                
                if distance < best_distance:
                    best_distance = distance
                    best_token = candidate_token
                
                if self.check_match(target_h_t, candidate_h):
                    found = True
                    recovered_tokens.append(candidate_token)
                    distances.append(distance)
                    prefix_ids = torch.cat([
                        prefix_ids,
                        torch.tensor([[candidate_token]], device=self.device)
                    ], dim=1)
                    if verbose:
                        token_str = self.tokenizer.decode([candidate_token])
                        print(f"  Found: '{token_str}' (d={distance:.6f})")
                    break
            
            if not found:
                if verbose:
                    token_str = self.tokenizer.decode([best_token]) if best_token else "None"
                    print(f"\n  No exact match at pos {t}, using best: '{token_str}' (d={best_distance:.4f})")
                recovered_tokens.append(best_token)
                distances.append(best_distance)
                prefix_ids = torch.cat([
                    prefix_ids,
                    torch.tensor([[best_token]], device=self.device)
                ], dim=1)
        
        return recovered_tokens, distances
    
    def invert_from_text(self, text: str, policy: str = "random", top_k: int = 256, verbose: bool = False):
        """Convenience method to invert text by first encoding it."""
        input_ids = self.tokenizer.encode(text, return_tensors="pt").to(self.device)
        
        if verbose:
            print(f"Extracting hidden states from layer {self.layer_idx}...")
        hidden_states = self.get_hidden_states(input_ids)
        
        if verbose:
            print(f"Inverting sequence of length {input_ids.shape[1]} tokens...")
            print(f"Policy: {policy}" + (f" (top_k={top_k})" if policy == "gradient" else ""))
        
        recovered_tokens, distances = self.invert_sequence(
            hidden_states, policy=policy, top_k=top_k, verbose=verbose
        )
        
        recovered_text = self.tokenizer.decode(recovered_tokens)
        avg_distance = np.mean(distances)
        
        return recovered_text, avg_distance

## Load Pre-Quantized Model with Unsloth

Using `unsloth/gemma-3-4b-it-unsloth-bnb-4bit` loaded via Unsloth's FastLanguageModel.
- Unsloth handles the bnb-4bit quantization state properly
- **VRAM**: ~2-3 GB for weights (4-bit) + activations

In [ ]:
# Pre-quantized Gemma 3 4B from Unsloth (bitsandbytes 4-bit)
# This is already quantized - no need for load_in_4bit flag
MODEL_NAME = "unsloth/gemma-3-4b-it-unsloth-bnb-4bit"

print(f"Using pre-quantized model: {MODEL_NAME}")

In [ ]:
# Initialize SipIt with pre-quantized Unsloth model
# layer_idx=0: first transformer layer (after embeddings)
# epsilon=1e-3: tolerance for matching hidden states

sipit = SipItGemma3(
    model_name=MODEL_NAME,
    layer_idx=0,
    epsilon=1e-3
)

## Test String Inversion

In [ ]:
# Simple test string
test_text = "Hello, world!"

print(f"Original text: '{test_text}'")
print(f"\nTokenizing...")

# Show tokenization
tokens = sipit.tokenizer.encode(test_text)
print(f"Token IDs: {tokens}")
print(f"Tokens: {[sipit.tokenizer.decode([t]) for t in tokens]}")
print(f"Sequence length: {len(tokens)}")

In [ ]:
# Run inversion with gradient-guided policy (much faster than random!)
print("Running inversion with GRADIENT-GUIDED policy...")
print("(Uses backprop to find top-k candidates, then searches those)\n")

recovered_text, avg_distance = sipit.invert_from_text(
    test_text,
    policy="gradient",
    top_k=256,  # Only test top 256 candidates per position
    verbose=True
)

print("\n" + "="*60)
print("RESULTS")
print("="*60)
print(f"Original:  '{test_text}'")
print(f"Recovered: '{recovered_text}'")
print(f"Average distance: {avg_distance:.6f}")
print(f"Perfect match: {test_text == recovered_text}")

## Compare Random vs Gradient-Guided Search

The gradient-guided policy uses backprop to find promising candidates:
- 1 forward+backward pass to get gradient direction
- Find top-k tokens aligned with negative gradient  
- Only evaluate those k candidates (vs 262k random)

**Expected speedup**: ~1000x fewer forward passes per position

In [ ]:
import time

test_text_compare = "Hello"

print(f"Test string: '{test_text_compare}'")
print(f"Tokens: {sipit.tokenizer.encode(test_text_compare)}")
print()

# Test gradient-guided (should be much faster)
print("=" * 60)
print("GRADIENT-GUIDED SEARCH (top_k=256)")
print("=" * 60)
start = time.time()
recovered_grad, dist_grad = sipit.invert_from_text(
    test_text_compare, 
    policy="gradient", 
    top_k=256,
    verbose=True
)
time_grad = time.time() - start
print(f"\nRecovered: '{recovered_grad}'")
print(f"Time: {time_grad:.2f}s")
print(f"Match: {test_text_compare == recovered_grad}")

print()

# Test random (baseline - may be slower)
print("=" * 60)
print("RANDOM SEARCH")
print("=" * 60)
start = time.time()
recovered_rand, dist_rand = sipit.invert_from_text(
    test_text_compare, 
    policy="random", 
    verbose=True
)
time_rand = time.time() - start
print(f"\nRecovered: '{recovered_rand}'")
print(f"Time: {time_rand:.2f}s")
print(f"Match: {test_text_compare == recovered_rand}")

print()
print("=" * 60)
print("COMPARISON")
print("=" * 60)
print(f"Gradient: {time_grad:.2f}s, dist={dist_grad:.6f}, match={test_text_compare == recovered_grad}")
print(f"Random:   {time_rand:.2f}s, dist={dist_rand:.6f}, match={test_text_compare == recovered_rand}")

## Manual Step-by-Step Inversion with Gradient Search

Let's break down the gradient-guided inversion process to see how it selects candidates.

In [ ]:
# Manual step-by-step process with gradient search
test_text3 = "Hi"

print(f"Test string: '{test_text3}'")

# Step 1: Encode
input_ids = sipit.tokenizer.encode(test_text3, return_tensors="pt").to(sipit.device)
print(f"\n1. Encoded token IDs: {input_ids.tolist()}")

# Step 2: Get hidden states (target)
hidden_states = sipit.get_hidden_states(input_ids)
print(f"2. Hidden states shape: {hidden_states.shape}")
print(f"   (batch_size, seq_len, hidden_dim)")

# Step 3: Show gradient-guided candidate selection for first position
print(f"\n3. Gradient-guided candidate selection for position 0:")
prefix = torch.empty((1, 0), dtype=torch.long, device=sipit.device)
target_h0 = hidden_states[:, 0:1, :]
top_candidates = sipit.gradient_guided_topk(prefix, target_h0, top_k=10)
print(f"   Top 10 candidates: {top_candidates}")
print(f"   As tokens: {[sipit.tokenizer.decode([t]) for t in top_candidates]}")

# Step 4: Full inversion
print(f"\n4. Full inversion with gradient policy...")
recovered_tokens, distances = sipit.invert_sequence(
    hidden_states,
    policy="gradient",
    top_k=256,
    verbose=False
)

print(f"   Recovered token IDs: {recovered_tokens}")
print(f"   Distances: {[f'{d:.6f}' for d in distances]}")

# Step 5: Decode
recovered = sipit.tokenizer.decode(recovered_tokens)
print(f"\n5. Decoded text: '{recovered}'")
print(f"   Match: {test_text3 == recovered}")

## Summary

In [ ]:
print("SipIt Inversion Test Complete!")
print(f"\nModel: {MODEL_NAME}")
print(f"Layer used: {sipit.layer_idx}")
print(f"Epsilon: {sipit.epsilon}")
print(f"Device: {sipit.device}")